In [ ]:
import os

dataset_root = "/kaggle/input/datasets/username/numerai-data-v52"
cus_feat_path = "/kaggle/input/datasets/username/numerai-custom-features/custom_features.json"
print(f"{dataset_root}")
print(f"{os.listdir(dataset_root)}, {cus_feat_path}")

/kaggle/input/datasets/shuangsong/numerai-data-v52
['features.json', 'train.parquet', 'validation.parquet'], /kaggle/input/datasets/shuangsong/numerai-custom-features/custom_features.json


In [ ]:
import pandas as pd
import numpy as np
import gc
import torch

def load_optimized(path, features, flag, tv_ratio=0.8):
    df = pd.read_parquet(path, columns=['target', 'era'] + features, engine='pyarrow')
    df = df.iloc[::4]

    if df['era'].dtype == 'object':
        df['era'] = df['era'].str.replace('era', '').astype(int)
    else:
        df['era'] = df['era'].astype(int)

    df = df.astype('float32')
    unique_eras = np.sort(df['era'].unique())
    ckpt_era = unique_eras[int(len(unique_eras) * tv_ratio)]

    if flag == "train":
        mask = df['era'] <= ckpt_era
    elif flag == "val":
        mask = df['era'] > ckpt_era
        
    X = torch.from_numpy(df.loc[mask, features].values).float()
    y = torch.from_numpy(df.loc[mask, 'target'].values).float()
    
    del df
    gc.collect() 
    
    return X, y

In [ ]:
%pip install catboost
from catboost import CatBoostRegressor

In [17]:
import numpy as np
import json
import torch

with open(cus_feat_path, 'r') as f:
    features = json.load(f)['feature_sets']['custom_features']

X_train, y_train = load_optimized(os.path.join(dataset_root, "train.parquet"), features, flag='train')
X_val, y_val = load_optimized(os.path.join(dataset_root, "validation.parquet"), features, flag='val')

train_mask = ~torch.isnan(y_train)
X_train_clean = X_train[train_mask]
y_train_clean = y_train[train_mask]

val_mask = ~torch.isnan(y_val)
X_val_clean = X_val[val_mask]
y_val_clean = y_val[val_mask]

cb_model = CatBoostRegressor(
    iterations=2000,      
    learning_rate=0.005,  
    depth=7,              
    l2_leaf_reg=5.0,
    random_strength=2.0,
    task_type="GPU",
    loss_function='RMSE',
    use_best_model=False,
    verbose=0             
)

cb_model.fit(
    X_train_clean.numpy(), y_train_clean.numpy(),
    eval_set=(X_val_clean.numpy(), y_val_clean.numpy()),
    early_stopping_rounds=500
)

val_preds = cb_model.predict(X_val_clean.numpy())
corr = pd.Series(y_val_clean.numpy()).corr(pd.Series(val_preds), method='spearman')

y_val_series = pd.Series(y_val_clean.numpy())
best_corr = -1
best_iter = 0

for i, staged_preds in enumerate(cb_model.staged_predict(X_val_clean.numpy())):
    corr = y_val_series.corr(pd.Series(staged_preds), method='spearman')
    print(f"Iteration {i}: Validation CORR = {corr:.4f}")
    
    if corr > best_corr:
        best_corr = corr
        best_iter = i

print("-" * 30)
print(f"Best CatBoost CORR: {best_corr:.4f} At Iteration {best_iter}")

corr_str = f"{best_corr:.4f}".split('.')[-1]
save_path = f"/kaggle/working/catboost_model_{corr_str}.cbm"
cb_model.save_model(save_path)

Iteration 0: Validation CORR = 0.0017
Iteration 1: Validation CORR = 0.0054
Iteration 2: Validation CORR = 0.0053
Iteration 3: Validation CORR = 0.0054
Iteration 4: Validation CORR = 0.0058
Iteration 5: Validation CORR = 0.0055
Iteration 6: Validation CORR = 0.0042
Iteration 7: Validation CORR = 0.0044
Iteration 8: Validation CORR = 0.0016
Iteration 9: Validation CORR = 0.0016
Iteration 10: Validation CORR = 0.0001
Iteration 11: Validation CORR = -0.0008
Iteration 12: Validation CORR = 0.0001
Iteration 13: Validation CORR = 0.0008
Iteration 14: Validation CORR = 0.0006
Iteration 15: Validation CORR = 0.0002
Iteration 16: Validation CORR = -0.0001
Iteration 17: Validation CORR = 0.0009
Iteration 18: Validation CORR = 0.0008
Iteration 19: Validation CORR = 0.0014
Iteration 20: Validation CORR = 0.0022
Iteration 21: Validation CORR = 0.0026
Iteration 22: Validation CORR = 0.0029
Iteration 23: Validation CORR = 0.0032
Iteration 24: Validation CORR = 0.0028
Iteration 25: Validation CORR = 0

In [19]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

train_preds_t = torch.tensor(cb_model.predict(X_train_clean.numpy()), dtype=torch.float32).unsqueeze(1)
val_preds_t = torch.tensor(val_preds, dtype=torch.float32).unsqueeze(1)

X_train_mlp = torch.cat([X_train_clean, train_preds_t], dim=1)
X_val_mlp = torch.cat([X_val_clean, val_preds_t], dim=1).cuda()

dataset = TensorDataset(X_train_mlp, y_train_clean)
loader = DataLoader(dataset, batch_size=8192, shuffle=True)

class NumeraiMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(0.1), 
            nn.Linear(in_dim, 256),
            nn.SiLU(),     
            nn.Linear(256, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze()

mlp = NumeraiMLP(X_train_mlp.shape[1]).cuda()
optimizer = optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=0.1)
criterion = nn.MSELoss()

best_corr = -1
best_epoch = 0
epochs = 100

prev_save_path = None

for epoch in range(epochs):
    mlp.train() 
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.cuda(), batch_y.cuda()
        optimizer.zero_grad()
        loss = criterion(mlp(batch_x), batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mlp.parameters(), 1.0) 
        optimizer.step()

    mlp.eval()
    with torch.no_grad():
        val_mlp_preds = mlp(X_val_mlp).cpu().numpy()

    corr = pd.Series(y_val_clean.numpy()).corr(pd.Series(val_mlp_preds), method='spearman')
    print(f"Epoch [{epoch+1}/{epochs}] Validation CORR: {corr:.4f}")
    
    if corr > best_corr:
        best_corr = corr
        best_epoch = epoch + 1
        corr_str = f"{best_corr:.4f}".split('.')[-1]
        save_path = f"/kaggle/working/mlp_model_{corr_str}.pth"
        
        if prev_save_path is not None and os.path.exists(prev_save_path):
            os.remove(prev_save_path)
            
        torch.save(mlp.state_dict(), save_path)
        prev_save_path = save_path

print("-" * 30)
print(f"Best MLP CORR: {best_corr:.4f} At Epoch {best_epoch}")

Epoch [1/100] Validation CORR: 0.0017
Epoch [2/100] Validation CORR: 0.0017
Epoch [3/100] Validation CORR: 0.0026
Epoch [4/100] Validation CORR: 0.0032
Epoch [5/100] Validation CORR: 0.0026
Epoch [6/100] Validation CORR: 0.0028
Epoch [7/100] Validation CORR: 0.0031
Epoch [8/100] Validation CORR: 0.0031
Epoch [9/100] Validation CORR: 0.0042
Epoch [10/100] Validation CORR: 0.0034
Epoch [11/100] Validation CORR: 0.0038
Epoch [12/100] Validation CORR: 0.0044
Epoch [13/100] Validation CORR: 0.0055
Epoch [14/100] Validation CORR: 0.0067
Epoch [15/100] Validation CORR: 0.0077
Epoch [16/100] Validation CORR: 0.0067
Epoch [17/100] Validation CORR: 0.0102
Epoch [18/100] Validation CORR: 0.0093
Epoch [19/100] Validation CORR: 0.0097
Epoch [20/100] Validation CORR: 0.0078
Epoch [21/100] Validation CORR: 0.0074
Epoch [22/100] Validation CORR: 0.0057
Epoch [23/100] Validation CORR: 0.0058
Epoch [24/100] Validation CORR: 0.0062
Epoch [25/100] Validation CORR: 0.0068
Epoch [26/100] Validation CORR: 0.